# WICOS - Radar Hand-Gesture Classification: Model Comparison
EE4C12 Machine Learning for Electrical Engineering

Jim van IJcken (5840597), Huib Maas (5858038)

Pre-processing: `notebooks/preprocessing.ipynb`, exported to `data/processed/wicos_dataset_jim.npz` by `src/export_jim_preprocessing.py`.

## Setup

In [ ]:
import os
import re
N_JOBS = 4                                    # CPU limit: full load on all 20 threads shut the laptop down
os.environ["OMP_NUM_THREADS"] = str(N_JOBS)   # set before numpy / TensorFlow are imported
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"      # hide TensorFlow info messages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn import svm
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.model_selection import LeaveOneGroupOut, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.config.experimental.enable_op_determinism()   # reproducible CNN training
tf.config.threading.set_intra_op_parallelism_threads(N_JOBS)
tf.config.threading.set_inter_op_parallelism_threads(2)

OUT_DIR = Path("../results/allmodels")      # all figures and CSV files of this notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Data

In [ ]:
# Pre-processed dataset (run src/prepare_data.py, then src/export_jim_preprocessing.py)
data = np.load("../data/processed/wicos_dataset_jim.npz", allow_pickle=True)

X_feat = data["X_feat"]                         # (960, 27) hand-crafted features
X_img = data["X_img"].reshape(960, -1)          # (960, 16384) flattened 128x128 spectrograms
y = data["y"]                                   # 0=wave, 1=pinch, 2=swipe, 3=click
persons = data["persons"]                       # person who performed the gesture (A-F)
gestures = list(data["gestures"])
idx_train, idx_test, cv_fold = data["idx_train"], data["idx_test"], data["cv_fold"]

print("Features:", X_feat.shape)
print("Images:  ", X_img.shape)
print("Train / test samples:", len(idx_train), "/", len(idx_test))

In [ ]:
# Validation schemes
# LOSO:   train on 5 persons, validate on the 6th (generalisation to a new person)
# Random: 5 random folds in the 80% training set; best model tested once on the 20% test set
loso_folds = list(LeaveOneGroupOut().split(X_feat, y, groups=persons))
random_folds = [(idx_train[cv_fold[idx_train] != k], idx_train[cv_fold[idx_train] == k]) for k in range(5)]
SCHEMES = [("LOSO", loso_folds), ("Random", random_folds)]

print("LOSO folds:  ", len(loso_folds), "- held-out persons:", [persons[val][0] for _, val in loso_folds])
print("Random folds:", len(random_folds))

In [ ]:
def evaluate(model, X, folds, n_jobs=N_JOBS):
    """Mean training and validation accuracy over the folds, and the validation accuracy per fold."""
    scores = cross_validate(model, X, y, cv=folds, scoring="accuracy", return_train_score=True, n_jobs=n_jobs)
    return scores["train_score"].mean(), scores["test_score"].mean(), scores["test_score"]


def best_setting(results, scheme):
    """Row of the tuning results with the highest mean validation accuracy."""
    r = results[results.scheme == scheme]
    return r.loc[r.val_acc.idxmax()]


def label(value):
    """Readable axis label for a hyperparameter value."""
    if isinstance(value, str):
        return value
    return "None" if pd.isna(value) else f"{value:g}"


def plot_tuning(results, x, hue=None, fixed=None, title="", filename=None):
    """Training (dashed) and validation (solid) accuracy versus hyperparameter x, for both schemes."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True, constrained_layout=True)
    for ax, (scheme, _) in zip(axes, SCHEMES):
        r = results[results.scheme == scheme]
        if fixed is not None:
            for key, value in fixed.items():
                r = r[r[key] == value]
        groups = r.groupby(hue, sort=False, dropna=False) if hue else [(None, r)]
        for i, (h, g) in enumerate(groups):
            xs = [label(v) for v in g[x]]
            name = f"{hue} = {label(h)}" if hue else "accuracy"
            ax.plot(xs, g.val_acc, "o-", color=f"C{i}", label=name)
            ax.plot(xs, g.train_acc, "--", color=f"C{i}", alpha=0.6)
        ax.set_title(f"{title} ({scheme})")
        ax.set_xlabel(x)
        ax.set_ylim(0.2, 1.02)
        ax.grid(alpha=0.3)
    axes[0].set_ylabel("Accuracy (solid: validation, dashed: training)")
    axes[1].legend(fontsize=8)
    if filename:
        fig.savefig(OUT_DIR / filename, bbox_inches="tight")
    plt.show()


def scores(y_true, y_pred):
    """Accuracy and macro-averaged precision, recall and F1 (every gesture counts equally)."""
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, average="macro"),
        "recall": recall_score(y_true, y_pred, average="macro"),
        "f1": f1_score(y_true, y_pred, average="macro"),
    }


def params(best):
    """Hyperparameters of a best-setting row as text, e.g. 'kernel=rbf, C=10'."""
    hyper = best.drop(["scheme", "train_acc", "val_acc", "fold_acc"])
    return ", ".join(f"{k}={label(v)}" for k, v in hyper.items())


def final_evaluation(name, model_loso, model_random, X, best_loso, best_random, n_jobs=N_JOBS):
    """LOSO predictions of the best LOSO model; test-set score of the best random-split model."""
    loso_pred = cross_val_predict(model_loso, X, y, cv=loso_folds, n_jobs=n_jobs)
    model_random.fit(X[idx_train], y[idx_train])
    test_pred = model_random.predict(X[idx_test])

    result = {
        "model": name,
        "LOSO accuracy": best_loso.val_acc,
        "LOSO std": np.std(best_loso.fold_acc, ddof=1),
        "LOSO macro-F1": f1_score(y, loso_pred, average="macro"),
        "Random CV accuracy": best_random.val_acc,
        "Test accuracy": accuracy_score(y[idx_test], test_pred),
        "Test macro-F1": f1_score(y[idx_test], test_pred, average="macro"),
        "LOSO params": params(best_loso),
        "Random params": params(best_random),
        "person_acc": best_loso.fold_acc,
        "loso_pred": loso_pred,
        "test_pred": test_pred,
    }
    print(f"{name}: LOSO accuracy {result['LOSO accuracy']:.3f} ± {result['LOSO std']:.3f} | "
          f"test accuracy (random split) {result['Test accuracy']:.3f}")
    return result


summary = []      # one result per model

## Logistic regression

In [ ]:
# Standardised features -> multinomial logistic regression (L2 regularisation, C = inverse strength)
def build_lr(C):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(C=C, max_iter=10000)),
    ])


lr_rows = []
for C in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:
    for scheme, folds in SCHEMES:
        train_acc, val_acc, fold_acc = evaluate(build_lr(C), X_feat, folds)
        lr_rows.append({"scheme": scheme, "C": C, "train_acc": train_acc, "val_acc": val_acc, "fold_acc": fold_acc})
lr_results = pd.DataFrame(lr_rows)

lr_best_loso = best_setting(lr_results, "LOSO")
lr_best_random = best_setting(lr_results, "Random")
print(f"Best C (LOSO):   {lr_best_loso.C}   validation accuracy {lr_best_loso.val_acc:.3f}")
print(f"Best C (Random): {lr_best_random.C}   validation accuracy {lr_best_random.val_acc:.3f}")

In [ ]:
plot_tuning(lr_results, x="C", title="Logistic regression", filename="lr_tuning.pdf")

In [ ]:
summary.append(final_evaluation("LR", build_lr(lr_best_loso.C), build_lr(lr_best_random.C), X_feat,
                                lr_best_loso, lr_best_random))

## SVM (features)

In [ ]:
# Standardised features -> soft-margin SVM; kernel sets the shape of the decision boundary
# Iteration limit: a linear SVM on some folds never converges (all other fits need < 2e6 iterations)
MAX_ITER = 2_000_000


def build_svm(kernel, C):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", svm.SVC(kernel=kernel, C=C, max_iter=MAX_ITER)),
    ])


svm_rows = []
for kernel in ["linear", "rbf", "poly"]:
    for C in [0.1, 1, 10, 100]:
        for scheme, folds in SCHEMES:
            train_acc, val_acc, fold_acc = evaluate(build_svm(kernel, C), X_feat, folds)
            svm_rows.append({"scheme": scheme, "kernel": kernel, "C": C,
                             "train_acc": train_acc, "val_acc": val_acc, "fold_acc": fold_acc})
svm_results = pd.DataFrame(svm_rows)

svm_best_loso = best_setting(svm_results, "LOSO")
svm_best_random = best_setting(svm_results, "Random")
print(f"Best (LOSO):   kernel {svm_best_loso.kernel}, C {svm_best_loso.C}   validation accuracy {svm_best_loso.val_acc:.3f}")
print(f"Best (Random): kernel {svm_best_random.kernel}, C {svm_best_random.C}   validation accuracy {svm_best_random.val_acc:.3f}")

In [ ]:
plot_tuning(svm_results, x="C", hue="kernel", title="SVM (features)", filename="svm_features_tuning.pdf")

In [ ]:
summary.append(final_evaluation("SVM (features)",
                                build_svm(svm_best_loso.kernel, svm_best_loso.C),
                                build_svm(svm_best_random.kernel, svm_best_random.C),
                                X_feat, svm_best_loso, svm_best_random))

## SVM + PCA (images)

In [ ]:
# PCA of all images (visualisation only; in the models PCA is fitted on the training folds)
pca = PCA(n_components=100, svd_solver="randomized", random_state=SEED)
X_pca = pca.fit_transform(X_img)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
axes[0].bar(np.arange(1, 21), pca.explained_variance_ratio_[:20], label="Per component")
axes[0].plot(np.arange(1, 21), np.cumsum(pca.explained_variance_ratio_[:20]), "o-", color="tab:orange", label="Cumulative")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Explained variance ratio")
axes[0].set_title("Explained variance")
axes[0].legend()
for j, g in enumerate(gestures):
    axes[1].scatter(X_pca[y == j, 0], X_pca[y == j, 1], s=8, alpha=0.6, label=g)
axes[1].set_xlabel("Component 1")
axes[1].set_ylabel("Component 2")
axes[1].set_title("First two principal components")
axes[1].legend()
fig.savefig(OUT_DIR / "pca_images.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# 16384 pixels vs 960 samples: PCA (whitened) reduces the images before the SVM
def build_svm_pca(n_components, kernel, C):
    return Pipeline([
        ("pca", PCA(n_components=n_components, whiten=True, svd_solver="randomized", random_state=SEED)),
        ("clf", svm.SVC(kernel=kernel, C=C, max_iter=MAX_ITER)),
    ])


pca_rows = []
for n_components in [20, 50, 100]:
    for kernel in ["linear", "rbf", "poly"]:
        for C in [0.1, 1, 10, 100]:
            for scheme, folds in SCHEMES:
                train_acc, val_acc, fold_acc = evaluate(build_svm_pca(n_components, kernel, C), X_img, folds)
                pca_rows.append({"scheme": scheme, "n_components": n_components, "kernel": kernel, "C": C,
                                 "train_acc": train_acc, "val_acc": val_acc, "fold_acc": fold_acc})
pca_results = pd.DataFrame(pca_rows)

pca_best_loso = best_setting(pca_results, "LOSO")
pca_best_random = best_setting(pca_results, "Random")
for scheme, b in [("LOSO", pca_best_loso), ("Random", pca_best_random)]:
    print(f"Best ({scheme}): {b.n_components} components, kernel {b.kernel}, C {b.C}   validation accuracy {b.val_acc:.3f}")

In [ ]:
plot_tuning(pca_results, x="C", hue="n_components", fixed={"kernel": pca_best_loso.kernel},
            title=f"SVM + PCA, {pca_best_loso.kernel} kernel", filename="svm_pca_tuning.pdf")

In [ ]:
summary.append(final_evaluation("SVM + PCA (images)",
                                build_svm_pca(pca_best_loso.n_components, pca_best_loso.kernel, pca_best_loso.C),
                                build_svm_pca(pca_best_random.n_components, pca_best_random.kernel, pca_best_random.C),
                                X_img, pca_best_loso, pca_best_random))

## Random forest

In [ ]:
# Bagged decision trees with majority vote; no scaling needed. max_depth=None: fully grown trees
def build_rf(n_estimators, max_depth):
    return RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=SEED)


def depth(value):
    # max_depth=None is stored as NaN in the results table
    return None if pd.isna(value) else int(value)


rf_rows = []
for n_estimators in [50, 100, 300]:
    for max_depth in [3, 5, 10, None]:
        for scheme, folds in SCHEMES:
            train_acc, val_acc, fold_acc = evaluate(build_rf(n_estimators, max_depth), X_feat, folds)
            rf_rows.append({"scheme": scheme, "n_estimators": n_estimators, "max_depth": max_depth,
                            "train_acc": train_acc, "val_acc": val_acc, "fold_acc": fold_acc})
rf_results = pd.DataFrame(rf_rows)

rf_best_loso = best_setting(rf_results, "LOSO")
rf_best_random = best_setting(rf_results, "Random")
for scheme, b in [("LOSO", rf_best_loso), ("Random", rf_best_random)]:
    print(f"Best ({scheme}): {b.n_estimators} trees, max depth {depth(b.max_depth)}   validation accuracy {b.val_acc:.3f}")

In [ ]:
plot_tuning(rf_results, x="max_depth", hue="n_estimators", title="Random forest", filename="rf_tuning.pdf")

In [ ]:
summary.append(final_evaluation("RF",
                                build_rf(rf_best_loso.n_estimators, depth(rf_best_loso.max_depth)),
                                build_rf(rf_best_random.n_estimators, depth(rf_best_random.max_depth)),
                                X_feat, rf_best_loso, rf_best_random))

## CNN (images)

**Search in two steps** (selection on the LOSO validation accuracy, the accuracy for a new person):

1. **Depth:** start at 4 convolution blocks and add one block at a time until the LOSO validation accuracy goes down; the best depth is kept. Filters per block: 8, 16, 32, then 32 for every extra block. At most 7 blocks (128 pixels halved 7 times = 1). Learning rate 0.03 and dropout 0.5 during this step.
2. **Hyperparameters:** learning rate and dropout of the selected depth.

In [ ]:
class CNN(ClassifierMixin, BaseEstimator):
    """Convolution blocks (3x3 conv + ReLU + 2x2 max pool) -> dropout -> softmax.

    filters: filters per block, e.g. "8-16-32" = 3 blocks. Cross-entropy loss, mini-batch SGD (batch 32).
    Early stopping on the loss of 10% of the training data (patience 10); the validation fold stays unseen.
    """
    def __init__(self, filters="8-16-32-32", learning_rate=0.03, dropout=0.5, epochs=100, patience=10):
        self.filters = filters
        self.learning_rate = learning_rate
        self.dropout = dropout
        self.epochs = epochs
        self.patience = patience

    def build(self):
        blocks = []
        for f in self.filters.split("-"):
            blocks += [layers.Conv2D(int(f), 3, padding="same", activation="relu"), layers.MaxPooling2D(2)]
        return keras.Sequential([keras.Input(shape=(128, 128, 1)), *blocks, layers.Flatten(),
                                 layers.Dropout(self.dropout), layers.Dense(4, activation="softmax")])

    def fit(self, X, y):
        keras.backend.clear_session()                                   # free memory of earlier networks
        keras.utils.set_random_seed(SEED)
        X = X.reshape(-1, 128, 128, 1)
        stop_set = np.random.default_rng(SEED).random(len(y)) < 0.1      # early-stopping set
        self.model_ = self.build()
        self.model_.compile(optimizer=keras.optimizers.SGD(learning_rate=self.learning_rate),
                            loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=self.patience, restore_best_weights=True)
        self.history_ = self.model_.fit(X[~stop_set], y[~stop_set], validation_data=(X[stop_set], y[stop_set]),
                                        epochs=self.epochs, batch_size=32, callbacks=[early_stop], verbose=0).history
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        return self.model_.predict(X.reshape(-1, 128, 128, 1), verbose=0).argmax(axis=1)


def layout(n_blocks):
    """Filters per block: 8, 16, 32, then 32 for every extra block (5 -> "8-16-32-32-32")."""
    return "-".join(str(min(8 * 2 ** i, 32)) for i in range(n_blocks))


# Size per depth: every block halves the feature map, so the dense layer gets fewer inputs
for n_blocks in range(4, 8):
    model = CNN(filters=layout(n_blocks)).build()
    flat = model.layers[-3].output.shape[-1]
    print(f"{n_blocks} blocks  {layout(n_blocks):20s} flatten {flat:5d}   parameters {model.count_params():6d}")

In [ ]:
# Step 1: depth. Add blocks until the LOSO validation accuracy goes down (slow: 11 networks per depth)
depth_rows = []
previous = 0
for n_blocks in range(4, 8):
    filters = layout(n_blocks)
    for scheme, folds in SCHEMES:
        train_acc, val_acc, fold_acc = evaluate(CNN(filters=filters), X_img, folds, n_jobs=1)
        depth_rows.append({"scheme": scheme, "n_blocks": n_blocks, "filters": filters,
                           "train_acc": train_acc, "val_acc": val_acc, "fold_acc": fold_acc})
        print(f"{n_blocks} blocks {scheme:6s} training accuracy {train_acc:.3f}   validation accuracy {val_acc:.3f}")
    loso_acc = depth_rows[-2]["val_acc"]                 # LOSO row of this depth
    if loso_acc < previous:
        print(f"LOSO accuracy went down at {n_blocks} blocks: stop")
        break
    previous = loso_acc
depth_results = pd.DataFrame(depth_rows)

best_depth = best_setting(depth_results, "LOSO")
cnn_filters = best_depth.filters
print(f"Selected depth: {best_depth.n_blocks} blocks ({cnn_filters})   LOSO validation accuracy {best_depth.val_acc:.3f}")

In [ ]:
# Step 2: learning rate and dropout of the selected depth (0.03 / 0.5 is reused from step 1)
cnn_rows = []
for learning_rate in [0.01, 0.03, 0.1]:
    for dropout in [0.25, 0.5]:
        for scheme, folds in SCHEMES:
            if learning_rate == 0.03 and dropout == 0.5:
                r = depth_results[(depth_results.filters == cnn_filters) & (depth_results.scheme == scheme)].iloc[0]
                train_acc, val_acc, fold_acc = r.train_acc, r.val_acc, r.fold_acc
            else:
                model = CNN(filters=cnn_filters, learning_rate=learning_rate, dropout=dropout)
                train_acc, val_acc, fold_acc = evaluate(model, X_img, folds, n_jobs=1)
            cnn_rows.append({"scheme": scheme, "filters": cnn_filters, "learning_rate": learning_rate, "dropout": dropout,
                             "train_acc": train_acc, "val_acc": val_acc, "fold_acc": fold_acc})
            print(f"lr {learning_rate}, dropout {dropout}, {scheme:6s}: validation accuracy {val_acc:.3f}")
cnn_results = pd.DataFrame(cnn_rows)

cnn_best_loso = best_setting(cnn_results, "LOSO")
cnn_best_random = best_setting(cnn_results, "Random")
for scheme, b in [("LOSO", cnn_best_loso), ("Random", cnn_best_random)]:
    print(f"Best ({scheme}): learning rate {b.learning_rate}, dropout {b.dropout}   validation accuracy {b.val_acc:.3f}")

depth_results.drop(columns="fold_acc").to_csv(OUT_DIR / "cnn_depth_search.csv", index=False)
cnn_results.drop(columns="fold_acc").to_csv(OUT_DIR / "cnn_hyperparameters.csv", index=False)

In [ ]:
plot_tuning(cnn_results, x="learning_rate", hue="dropout", title=f"CNN, {best_depth.n_blocks} blocks", filename="cnn_tuning.pdf")

In [ ]:
# CNN search: (a) number of convolution blocks, (b) learning rate and dropout of the selected depth
with plt.rc_context({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False}):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), constrained_layout=True, gridspec_kw={"width_ratios": [1.25, 1]})

    # (a) Depth: validation accuracy (bars) and training accuracy (black lines) for both schemes
    ax = axes[0]
    val = depth_results.pivot(index="n_blocks", columns="scheme", values="val_acc")
    train = depth_results.pivot(index="n_blocks", columns="scheme", values="train_acc")
    blocks = val.index.to_numpy()
    x = np.arange(len(blocks))
    for k, (scheme, _) in enumerate(SCHEMES):
        pos = x + (k - 0.5) * 0.36
        ax.bar(pos, val[scheme], width=0.34, color=f"C{k}", label=f"Validation ({scheme})")
        ax.scatter(pos, train[scheme], marker="_", s=250, linewidths=1.5, color="black",
                   label="Training" if k == 0 else None, zorder=3)
        for p, v in zip(pos, val[scheme]):
            ax.text(p, v + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=7)
    selected = int(np.flatnonzero(blocks == best_depth.n_blocks)[0])
    ax.axvspan(selected - 0.45, selected + 0.45, color="gold", alpha=0.25, zorder=0, label="Selected")
    n_params = [CNN(filters=layout(b)).build().count_params() for b in blocks]
    ax.set_xticks(x, [f"{b} blocks\n{p / 1000:.1f}k parameters" for b, p in zip(blocks, n_params)])
    ax.set_ylim(0, 1.08)
    ax.set_ylabel("Accuracy")
    ax.set_title("(a) Number of convolution blocks (learning rate 0.03, dropout 0.5)")
    ax.legend(fontsize=7, loc="lower left", ncol=2)

    # (b) Learning rate and dropout: LOSO validation (solid) and training (dashed)
    ax = axes[1]
    loso = cnn_results[cnn_results.scheme == "LOSO"]
    for i, (dropout, g) in enumerate(loso.groupby("dropout")):
        lr = [label(v) for v in g.learning_rate]
        ax.plot(lr, g.val_acc, "o-", color=f"C{i + 2}", label=f"Validation, dropout {dropout:g}")
        ax.plot(lr, g.train_acc, "--", color=f"C{i + 2}", alpha=0.6, label=f"Training, dropout {dropout:g}")
    ax.scatter(label(cnn_best_loso.learning_rate), cnn_best_loso.val_acc, s=180, facecolors="none",
               edgecolors="black", linewidths=1.5, label="Selected", zorder=3)
    ax.set_ylim(0, 1.08)
    ax.set_xlabel("Learning rate")
    ax.set_title(f"(b) Learning rate and dropout ({best_depth.n_blocks} blocks, LOSO)")
    ax.legend(fontsize=7, loc="lower right")

    fig.savefig(OUT_DIR / "cnn_search.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
cnn_loso = CNN(filters=cnn_filters, learning_rate=cnn_best_loso.learning_rate, dropout=cnn_best_loso.dropout)
cnn_random = CNN(filters=cnn_filters, learning_rate=cnn_best_random.learning_rate, dropout=cnn_best_random.dropout)
summary.append(final_evaluation("CNN (images)", cnn_loso, cnn_random, X_img, cnn_best_loso, cnn_best_random, n_jobs=1))

In [ ]:
# Loss and accuracy per epoch of the final random-split CNN
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), constrained_layout=True)
for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(cnn_random.history_[metric], label="Training")
    ax.plot(cnn_random.history_["val_" + metric], label="Early-stopping set")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f"CNN {metric}")
axes[0].legend()
fig.savefig(OUT_DIR / "cnn_training_curve.pdf", bbox_inches="tight")
plt.show()

## Comparison

In [ ]:
# LOSO accuracy = best mean validation accuracy of the tuning loop (not nested, slightly optimistic)
comparison = pd.DataFrame(summary)
comparison.drop(columns=["person_acc", "loso_pred", "test_pred"]).round(3)

In [ ]:
# Random split vs LOSO per model
x = np.arange(len(comparison))
plt.figure(figsize=(9, 4))
plt.bar(x - 0.2, comparison["Test accuracy"], width=0.4, label="Random split (test set)")
plt.bar(x + 0.2, comparison["LOSO accuracy"], width=0.4, yerr=comparison["LOSO std"], capsize=4, label="LOSO (mean ± std)")
plt.axhline(0.25, color="gray", ls=":", label="Chance level")
plt.xticks(x, comparison.model)
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Random split vs leave-one-subject-out")
plt.legend(loc="lower right")
plt.savefig(OUT_DIR / "comparison_loso_vs_random.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Accuracy per held-out person
per_person = pd.DataFrame(list(comparison.person_acc), index=comparison.model,
                          columns=[persons[val][0] for _, val in loso_folds]).T
ax = per_person.plot.bar(figsize=(10, 4), width=0.8, rot=0)
ax.axhline(0.25, color="gray", ls=":")
ax.set_ylim(0, 1)
ax.set_xlabel("Held-out person")
ax.set_ylabel("Accuracy")
ax.set_title("LOSO accuracy per person")
ax.legend(ncol=3, fontsize=8, loc="lower right")
plt.savefig(OUT_DIR / "comparison_per_person.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Confusion matrix of the best setting of every model:
# left = LOSO predictions summed over the six held-out persons, right = random split, test set
for _, row in comparison.iterrows():
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), constrained_layout=True)
    ConfusionMatrixDisplay.from_predictions(y, row.loso_pred, display_labels=gestures, ax=axes[0], colorbar=False)
    ConfusionMatrixDisplay.from_predictions(y[idx_test], row.test_pred, display_labels=gestures, ax=axes[1], colorbar=False)
    axes[0].set_title(f"LOSO\n{row['LOSO params']}", fontsize=10)
    axes[1].set_title(f"Random split (test set)\n{row['Random params']}", fontsize=10)
    fig.suptitle(row.model)
    fig.savefig(OUT_DIR / f"confusion_{re.sub(r'[^a-z0-9]+', '_', row.model.lower()).strip('_')}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# Accuracy, precision, recall and F1 (macro) of the best setting of every model
metric_rows = []
for _, row in comparison.iterrows():
    metric_rows.append({"model": row.model, "validation": "LOSO", "best parameters": row["LOSO params"],
                        **scores(y, row.loso_pred)})
    metric_rows.append({"model": row.model, "validation": "Random split (test set)", "best parameters": row["Random params"],
                        **scores(y[idx_test], row.test_pred)})
metrics = pd.DataFrame(metric_rows)
metrics.round(3)

In [ ]:
# Save the results
metrics.to_csv(OUT_DIR / "metrics.csv", index=False)
comparison.drop(columns=["person_acc", "loso_pred", "test_pred"]).to_csv(OUT_DIR / "comparison.csv", index=False)
per_person.to_csv(OUT_DIR / "per_person_accuracy.csv", index_label="person")